[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [asyncpg and psycopg3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/asyncpg-and-psycopg3-deep-dive.html)

# JSONB &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell is the notebook's Setup, with five thousand events and the `docs` table. Run it
first. Task 5 makes an index and drops it again, so run it whole.


In [1]:
import getpass
import json
import os
import subprocess
import sys
import time
from importlib.metadata import PackageNotFoundError, version

try:
    if version("psycopg") < "3.3" or version("asyncpg") < "0.31":
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "psycopg[binary,pool]==3.3.6", "psycopg-pool==3.3.2", "asyncpg==0.31.0"],
                   check=True)

import asyncpg
import psycopg
from psycopg import errors
from psycopg.types.json import Json, Jsonb

def shell(command):
    """Run a shell command and hand back what it printed, without letting it stop the notebook."""
    done = subprocess.run(command, shell=True, capture_output=True, text=True)
    return done.returncode, (done.stdout + done.stderr).strip()


def answering(database="postgres"):
    """Whether a server is there, asked the only way that needs no client binaries."""
    try:
        with psycopg.connect(f"dbname={database}", connect_timeout=2):
            return True
    except psycopg.OperationalError:
        return False


def start_server(wait=60):
    """Install and start PostgreSQL if nothing is answering. Returns what it had to do."""
    if answering():
        return "already running"
    if sys.platform != "linux":
        raise RuntimeError("No PostgreSQL is answering. Start your own server and run this again: "
                           "this cell only installs one on Linux, which is what Colab runs.")

    sudo = "" if os.geteuid() == 0 else "sudo "
    shell(f"{sudo}apt-get -qq update")
    shell(f"{sudo}apt-get -qq -y install postgresql postgresql-contrib")
    shell(f"{sudo}service postgresql start")                        # Colab has no systemd

    for attempt in range(1, wait + 1):                              # start returns before it listens
        if shell("pg_isready -q")[0] == 0:
            break
        print(f"  waiting for the cluster ({attempt})")              # a silent minute looks hung
        time.sleep(1)
    else:
        raise RuntimeError(f"PostgreSQL did not accept connections within {wait} seconds.")

    me = getpass.getuser()                                          # peer authentication wants a role
    asking = f"""sudo -u postgres psql -tAc "SELECT 1 FROM pg_roles WHERE rolname='{me}'" """
    if shell(asking)[1] != "1":                                     # named for the operating system user
        shell(f"sudo -u postgres createuser -s {me}")
    return "installed and started"

def build(rows=5000):
    """Make the guide database and its events table, and fill it once."""
    with psycopg.connect("dbname=postgres", autocommit=True) as conn:
        if not conn.execute("SELECT 1 FROM pg_database WHERE datname = 'guide'").fetchone():
            conn.execute("CREATE DATABASE guide")                   # cannot run in a transaction

    with psycopg.connect("dbname=guide", autocommit=True) as conn:
        for (leftover,) in conn.execute(                            # whatever an earlier run made
                "SELECT tablename FROM pg_tables "
                "WHERE schemaname = 'public' AND tablename <> 'events'").fetchall():
            conn.execute(f'DROP TABLE IF EXISTS "{leftover}" CASCADE')

        conn.execute("""CREATE TABLE IF NOT EXISTS events (
                            id bigserial PRIMARY KEY,
                            ts timestamptz NOT NULL DEFAULT now(),
                            kind text NOT NULL,
                            payload jsonb NOT NULL)""")
        if conn.execute("SELECT count(*) FROM events").fetchone()[0] == 0:
            conn.execute("""INSERT INTO events (kind, payload)
                            SELECT (ARRAY['click', 'view', 'purchase'])[1 + n %% 3],
                                   jsonb_build_object('n', n, 'size', 1 + n %% 7)
                            FROM generate_series(1, %s) AS n""", (rows,))
        return conn.execute("SELECT count(*) FROM events").fetchone()[0]

def report():
    """One line naming what this notebook is running against."""
    rows = build()                                                  # makes the database if it is new
    with psycopg.connect("dbname=guide") as conn:
        major = int(conn.execute("SHOW server_version_num").fetchone()[0]) // 10000
    return (f"PostgreSQL {major} | psycopg {version('psycopg')} | asyncpg {version('asyncpg')} "
            f"| events: {rows} rows")

def nodes(plan, depth=0):
    """The plan as a list of node names, without the costs and timings that differ per machine."""
    lines = ["  " * depth + plan["Node Type"]]
    for child in plan.get("Plans", []):
        lines += nodes(child, depth + 1)
    return lines


def explain(conn, query, params=None):
    """Print how the server would run a query, and how many rows it expects."""
    plan = conn.execute("EXPLAIN (FORMAT JSON) " + query, params).fetchone()[0][0]["Plan"]
    for line in nodes(plan):
        print("   ", line)
    print("    expecting about", plan["Plan Rows"], "rows")


def build_docs():
    """One table with the same document in a jsonb column and in a json column."""
    with psycopg.connect("dbname=guide", autocommit=True) as conn:
        conn.execute("DROP TABLE IF EXISTS docs")
        conn.execute("CREATE TABLE docs (id serial PRIMARY KEY, body jsonb, plain json)")
        conn.execute("INSERT INTO docs (body, plain) VALUES (%s, %s)",
                     (Jsonb({"kind": "click", "size": 3, "tags": ["a", "b"]}),
                      Json({"kind": "click", "size": 3, "tags": ["a", "b"]})))


print("server:", start_server())
print(report())
build_docs()
print("docs is ready")


server: already running
PostgreSQL 16 | psycopg 3.3.6 | asyncpg 0.31.0 | events: 5000 rows
docs is ready


**1.** A dictionary, refused and then accepted.


In [2]:
document = {"size": 3}

with psycopg.connect("dbname=guide") as conn:
    try:
        conn.execute("SELECT %s", (document,))
    except psycopg.ProgrammingError as error:
        print("bare:  ", error)
    conn.rollback()

    print("Jsonb: ", conn.execute("SELECT pg_typeof(%s)::text", (Jsonb(document),)).fetchone()[0])
    print("Json:  ", conn.execute("SELECT pg_typeof(%s)::text", (Json(document),)).fetchone()[0])


bare:   cannot adapt type 'dict' using placeholder '%s' (format: AUTO)
Jsonb:  jsonb
Json:   json


The wrapper is the only difference, and it decides which of the two PostgreSQL types the value
becomes. `Jsonb` is the one to reach for, because the operators worth having belong to it.


**2.** Containment, one key and then two.


In [3]:
with psycopg.connect("dbname=guide") as conn:
    for wanted in ({"size": 5}, {"n": 4}, {"size": 5, "n": 4}):
        found = conn.execute("SELECT count(*) FROM events WHERE payload @> %s",
                             (Jsonb(wanted),)).fetchone()[0]
        print(f"  @> {str(wanted):<24} {found:>5}")


  @> {'size': 5}                714
  @> {'n': 4}                     1
  @> {'size': 5, 'n': 4}          1


One key matches everything that has it, and adding a second narrows to the documents that have both.
That is containment: a question about whether one document is inside another, not about text.


**3.** One value, three ways.


In [4]:
with psycopg.connect("dbname=guide") as conn:
    row = conn.execute("""
        SELECT payload -> 'size', payload ->> 'size', (payload ->> 'size')::int
        FROM events ORDER BY id LIMIT 1""").fetchone()

for label, value in zip(("-> 'size'", "->> 'size'", "cast to int"), row):
    print(f"  {label:<12} {type(value).__name__:<5} {value!r}")


  -> 'size'    int   2
  ->> 'size'   str   '2'
  cast to int  int   2


`->` keeps it JSON, `->>` makes it text, and the cast makes it a number. Which one you want depends
on what happens next: comparing against a number needs the third, because JSON has no `>` against an
integer and text compares as text.


**4.** The operator the json column does not have.


In [5]:
with psycopg.connect("dbname=guide") as conn:
    print("on the jsonb column:", conn.execute(
        "SELECT count(*) FROM docs WHERE body @> %s", (Jsonb({"size": 3}),)).fetchone()[0])

    try:
        conn.execute("SELECT count(*) FROM docs WHERE plain @> %s", (Jsonb({"size": 3}),))
    except errors.UndefinedFunction as error:
        print("on the json column: ", str(error).splitlines()[0])


on the jsonb column: 1
on the json column:  operator does not exist: json @> jsonb


Same document, same query, different column type. `json` keeps the text and `jsonb` keeps the
structure, and only the structure can be asked about.


**5.** The plan, before and after.


In [6]:
query = "SELECT id FROM events WHERE payload @> %s"
wanted = (Jsonb({"size": 5}),)

with psycopg.connect("dbname=guide", autocommit=True) as conn:
    conn.execute("DROP INDEX IF EXISTS events_size_idx")
    print("without an index:")
    explain(conn, query, wanted)

    conn.execute("CREATE INDEX events_size_idx ON events USING GIN (payload)")
    conn.execute("ANALYZE events")
    print("with one:")
    explain(conn, query, wanted)

    conn.execute("DROP INDEX events_size_idx")


without an index:
    Seq Scan
    expecting about 707 rows
with one:
    Bitmap Heap Scan
      Bitmap Index Scan
    expecting about 707 rows


`Seq Scan` reads every row and tests it. `Bitmap Index Scan` reads the index to decide which rows are
worth fetching, and `Bitmap Heap Scan` fetches those. The rows returned are identical either way,
which is why this difference has to be looked for rather than noticed.


**6.** The same question through asyncpg.


In [7]:
conn = await asyncpg.connect(database="guide")
await conn.set_type_codec("jsonb", encoder=json.dumps, decoder=json.loads, schema="pg_catalog")

found = await conn.fetchval("SELECT count(*) FROM events WHERE payload @> $1", {"size": 5})
row = await conn.fetchrow("SELECT id, payload FROM events WHERE payload @> $1 ORDER BY id", {"size": 5})

print("matching:", found)
print("one of them:", dict(row))
await conn.close()


matching: 714
one of them: {'id': 4, 'payload': {'n': 4, 'size': 5}}


With the codec registered there is no wrapper anywhere: a dictionary is the parameter and a
dictionary is what comes back. psycopg asks per value and asyncpg asks once per connection, which is
the same decision made in two different places.


---

&#8592; **Back to:** [JSONB](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/asyncpg-and-psycopg3-deep-dive/06-jsonb.ipynb)  &nbsp;&middot;&nbsp;  [asyncpg and psycopg3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/asyncpg-and-psycopg3-deep-dive.html)
